# `RunnableSerializable: Serializable, Runnable[Input, Output]`

`RunnableSerializable` is an abstract `Runnable` base class that supports LangChain JSON serialization and runtime-configurable fields or alternatives.

A concrete subclass must still provide the required Runnable execution implementation, such as `invoke()`.

## Type Parameters

```python
Input # Input type accepted by the Runnable
Output # Output type produced by the Runnable
```

## Field

```python
name: str | None = None # Optional name used for tracing, debugging, and serialization
```

## Constructor

The constructor is generated from the serializable model fields.

```python
RunnableSerializable(
    *,
    name: str | None = None, # Optional Runnable name
    **subclass_fields: Any, # Additional fields declared by the concrete subclass
) -> None # Initialize the serializable Runnable model
```

`RunnableSerializable` is normally subclassed rather than instantiated directly.

## Overridden Method

### `to_json`

Serializes the Runnable into LangChain's JSON-compatible constructor representation.

The generated Runnable name is added to the serialized result when possible.

## Methods

### `configurable_fields`

Marks selected model fields as values that may be changed through runtime configuration.

```python
configurable_fields(
    self, # RunnableSerializable instance
    **kwargs: AnyConfigurableField, # Model field names mapped to configurable-field definitions
) -> RunnableSerializable[Input, Output] # Return a new Runnable with configurable fields
```

It raises `ValueError` when a supplied field name is not declared by the Runnable model.

The original Runnable is not modified.

### `configurable_alternatives`

Creates a configurable Runnable that can select between multiple Runnable implementations at runtime.

```python
configurable_alternatives(
    self, # RunnableSerializable instance used as the default alternative
    which: ConfigurableField, # Configuration field used to select an alternative
    *,
    default_key: str = "default", # Key representing the current Runnable
    prefix_keys: bool = False, # Whether alternative configuration keys include the field identifier
    **kwargs: Runnable[Input, Output] | Callable[[], Runnable[Input, Output]], # Alternative keys mapped to Runnables or lazy factories
) -> RunnableSerializable[Input, Output] # Return a configurable Runnable containing the alternatives
```

Callable alternatives are created lazily when selected.

The original Runnable is used as the default implementation.

## Inherited Runnable Capabilities

Concrete subclasses inherit the normal Runnable API, including:

- Invocation and asynchronous invocation
- Batch and asynchronous batch execution
- Streaming and asynchronous streaming
- Runnable composition
- Input, output, and configuration schemas
- Runtime configuration
- Tracing and lifecycle listeners
- Retry and fallback wrappers

## Behaviour

- The class combines the `Serializable` and `Runnable` contracts.
- Concrete subclasses may declare additional serializable model fields.
- `name` is included in serialized output when possible.
- Runtime configurability creates wrapper Runnables instead of modifying the original object.
- Only declared model fields can be exposed through `configurable_fields()`.

In [ ]:
from langchain_core.runnables import RunnableConfig, RunnableSerializable # Import the required LangChain classes

class PrefixRunnable(RunnableSerializable[str, str]): # Create a serializable Runnable accepting and returning strings
    prefix: str # Declare a serializable model field

    def invoke( # Implement the required synchronous execution method
        self, # Current PrefixRunnable instance
        input: str, # Input text passed to the Runnable
        config: RunnableConfig | None = None, # Optional runtime configuration
    ) -> str: # Return the processed string
        return self.prefix + input # Add the stored prefix and return the result

runnable = PrefixRunnable(prefix="Result: ") # Create the Runnable with a serializable field

result = runnable.invoke("Hello LangChain") # Execute the Runnable

serialized_data = runnable.to_json() # Convert the Runnable into LangChain's serialized representation

print(result) # Display the execution result

print(serialized_data) # Display the serialized Runnable data